In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from tkinter import filedialog
from tkinter import Tk

def read_binary_file(filepath, shape, dtype=np.float64):
    """Read binary file and return as numpy array with the specified shape and dtype."""
    data = np.fromfile(filepath, dtype=dtype)
    return data.reshape(shape, order='F')

def process_files(data_folder, files):
    summary_data = {}

    for file in files:
        data_file = os.path.join(data_folder, file)
        data_double_file = os.path.splitext(data_file)[0]  # Replace '.mat' with ''

        # Check if the binary file exists
        if not os.path.isfile(data_double_file):
            print(f'Warning: The corresponding binary file for {file} was not found. Skipping this file.')
            continue

        # Load the .mat file
        mat_data = loadmat(data_file)
        allRandomizedStimOrders = mat_data['allRandomizedStimOrders']
        variables = mat_data['variables']

        # Load the binary data
        data_shape = (9, -1)  # Assuming (9, inf) as in MATLAB
        try:
            Data = read_binary_file(data_double_file, shape=(9, -1))
        except Exception as e:
            print(f'Failed to open the binary data file: {data_double_file}. Skipping this file. Error: {e}')
            continue

        # Extract variables
        fs = variables['SampleRate'][0][0]
        blocks = variables['blocks'][0][0]
        pre_time = 0.8
        post_time = 10
        pre_samples = int(pre_time * fs)
        post_samples = int(post_time * fs)
        pulse_interval = int(fs / variables['Frequency'][0][0])
        smooth_window = 100

        # Step 1: Identify all potential LED ON events for non-zero stimuli
        led_on_indices = []
        for k in range(len(Data[6, :]) - pulse_interval):
            if Data[6, k] > 9 and np.all(Data[6, k-pulse_interval:k] < 1) and np.any(Data[6, k:k+pulse_interval] > 9):
                led_on_indices.append(k)

        # Step 2: Match LED ON events with corresponding stim durations, handle 0 ms separately
        event_index = 0
        stimulus_indices = [None] * blocks
        stimulus_durations = [None] * blocks

        for block in range(blocks):
            randomizedOrder = allRandomizedStimOrders[block].flatten()
            num_stims = len(randomizedOrder)
            block_indices = []

            for stimIdx in range(num_stims):
                currentStimDuration = randomizedOrder[stimIdx]

                if currentStimDuration > 0:
                    # Handle non-zero stimuli
                    if event_index < len(led_on_indices):
                        block_indices.append(led_on_indices[event_index])
                        event_index += 1
                    else:
                        print(f'Warning: Not enough LED ON events detected for block {block + 1}')
                else:
                    # Handle 0 ms stimuli (no actual LED ON event)
                    expected_index = int((stimIdx * variables['TrialLength'][0][0] / num_stims) * fs + pre_samples)
                    block_indices.append(expected_index)

            stimulus_indices[block] = np.array(block_indices)
            stimulus_durations[block] = randomizedOrder

        # Step 3: Group events by stimulus duration across all blocks
        uniqueStimDurations = np.unique(np.concatenate(stimulus_durations))
        grouped_wbf_segments = {dur: [] for dur in uniqueStimDurations}
        grouped_block_traces = {dur: [] for dur in uniqueStimDurations}

        for j, currentStimDuration in enumerate(uniqueStimDurations):
            for block in range(blocks):
                indices_for_duration = stimulus_indices[block][stimulus_durations[block] == currentStimDuration]
                block_wbf_segments = []

                for i in indices_for_duration:
                    start_idx = max(i - pre_samples, 0)
                    end_idx = min(i + post_samples, Data.shape[1])

                    # Extract and smooth the WBF data
                    wbf_segment = np.floor(Data[3, start_idx:end_idx] * 100)  # WBF
                    smoothed_wbf = np.convolve(wbf_segment, np.ones(smooth_window)/smooth_window, mode='same')

                    # Adjust the segment by subtracting the baseline (mean WBF 0.5 sec before LED ON)
                    baseline_wbf = np.mean(smoothed_wbf[:min(pre_samples, len(smoothed_wbf))])
                    adjusted_wbf = smoothed_wbf - baseline_wbf

                    # Pad the segment if it's shorter than expected
                    segment_length = post_samples + pre_samples + 1
                    if len(adjusted_wbf) < segment_length:
                        adjusted_wbf = np.concatenate([adjusted_wbf, np.full(segment_length - len(adjusted_wbf), np.nan)])

                    # Store the adjusted segment in the matrix
                    block_wbf_segments.append(adjusted_wbf)

                grouped_block_traces[currentStimDuration].append(block_wbf_segments)
                grouped_wbf_segments[currentStimDuration].extend(block_wbf_segments)

        # Step 4: Plot the individual data
        fileSummary = {}

        for currentStimDuration in uniqueStimDurations:
            all_wbf_segments = np.array(grouped_wbf_segments[currentStimDuration])

            if all_wbf_segments.size > 0:
                # Calculate the mean WBF trace across all blocks
                mean_wbf = np.nanmean(all_wbf_segments, axis=0)

                # Store the mean_wbf data for summary plot
                fileSummary[f'stim_{int(currentStimDuration)}'] = mean_wbf

                # Create time axis for plotting
                time_axis = np.linspace(-pre_time, post_time, len(mean_wbf))

                # Plotting
                plt.figure()
                plt.ylim([-80, 10])
                plt.gca().set_facecolor('k')
                plt.gca().spines['bottom'].set_color('white')
                plt.gca().spines['top'].set_color('white')
                plt.gca().spines['right'].set_color('white')
                plt.gca().spines['left'].set_color('white')
                plt.gca().tick_params(axis='x', colors='white')
                plt.gca().tick_params(axis='y', colors='white')

                # Plot traces from each block
                for block in range(blocks):
                    block_wbf = np.array(grouped_block_traces[currentStimDuration][block])
                    if block_wbf.size > 0:
                        block_mean_wbf = np.nanmean(block_wbf, axis=0)
                        plt.plot(time_axis, block_mean_wbf, linewidth=1, color=[0, 0.5, 1, 0.3])

                # Plot the mean WBF trace across all blocks with thicker line
                plt.plot(time_axis, mean_wbf, linewidth=2, color=[0, 0.5, 1])

                # Shade the stim_length region in Bright Crimson
                stim_start_time = 0
                stim_end_time = stim_start_time + (currentStimDuration / 1000)
                plt.axvspan(stim_start_time, stim_end_time, color=[1, 0.196, 0.353], alpha=0.3)

                # Set labels and title
                plt.ylabel('WBF (Hz)', color='white')
                plt.xlabel('Time (s)', color='white')
                plt.title(f'Stim Duration: {int(currentStimDuration)} ms', color='white')

                # Save the figure
                save_dir = 'D:\\Yichen\\Plots'
                os.makedirs(save_dir, exist_ok=True)
                file_name = f'{os.path.splitext(file)[0]}_WBF_plot_{int(currentStimDuration)}ms.png'
                plt.savefig(os.path.join(save_dir, file_name), dpi=300, bbox_inches='tight', facecolor='black')
                plt.close()

        # Store this file's summary data
        summary_data[file] = fileSummary

    return summary_data

def create_summary_plots(summary_data, uniqueStimDurations):
    # Create summary plots across all files
    for currentStimDuration in uniqueStimDurations:
        plt.figure()
        plt.ylim([-80, 10])
        plt.gca().set_facecolor('k')
        plt.gca().spines['bottom'].set_color('white')
        plt.gca().spines['top'].set_color('white')
        plt.gca().spines['right'].set_color('white')
        plt.gca().spines['left'].set_color('white')
        plt.gca().tick_params(axis='x', colors='white')
        plt.gca().tick_params(axis='y', colors='white')

        time_axis = np.linspace(-pre_time, post_time, len(next(iter(summary_data.values()))[f'stim_{int(currentStimDuration)}']))

        # Plot individual files' data
        for file, file_data in summary_data.items():
            if f'stim_{int(currentStimDuration)}' in file_data:
                file_wbf = file_data[f'stim_{int(currentStimDuration)}']
                plt.plot(time_axis, file_wbf, linewidth=1, color=[0, 0.5, 1, 0.3])

        # Calculate and plot the grand average across all files
        grand_average = np.zeros_like(time_axis)
        count = 0
        for file, file_data in summary_data.items():
            if f'stim_{int(currentStimDuration)}' in file_data:
                grand_average += file_data[f'stim_{int(currentStimDuration)}']
                count += 1
        if count > 0:
            grand_average /= count
            plt.plot(time_axis, grand_average, linewidth=2, color=[0, 0.5, 1])

        # Shade the stim_length region in Bright Crimson
        stim_start_time = 0
        stim_end_time = stim_start_time + (currentStimDuration / 1000)
        plt.axvspan(stim_start_time, stim_end_time, color=[1, 0.196, 0.353], alpha=0.3)

        # Set labels and title
        plt.ylabel('WBF (Hz)', color='white')
        plt.xlabel('Time (s)', color='white')
        plt.title(f'Summary: Stim Duration {int(currentStimDuration)} ms', color='white')

        # Save the summary figure
        save_dir = 'D:\\Yichen\\Plots'
        summary_file_name = f'Summary_WBF_plot_{int(currentStimDuration)}ms.png'
        plt.savefig(os.path.join(save_dir, summary_file_name), dpi=300, bbox_inches='tight', facecolor='black')
        plt.close()

# Final summary plot with all stim duration averages from all files
def create_final_summary_plot(summary_data, uniqueStimDurations):
    plt.figure()
    plt.ylim([-80, 10])
    plt.gca().set_facecolor('k')
    plt.gca().spines['bottom'].set_color('white')
    plt.gca().spines['top'].set_color('white')
    plt.gca().spines['right'].set_color('white')
    plt.gca().spines['left'].set_color('white')
    plt.gca().tick_params(axis='x', colors='white')
    plt.gca().tick_params(axis='y', colors='white')

    colors = plt.cm.get_cmap('tab10', len(uniqueStimDurations))

    for j, currentStimDuration in enumerate(uniqueStimDurations):
        grand_average = np.zeros_like(time_axis)
        count = 0

        for file, file_data in summary_data.items():
            if f'stim_{int(currentStimDuration)}' in file_data:
                grand_average += file_data[f'stim_{int(currentStimDuration)}']
                count += 1
        if count > 0:
            grand_average /= count
            plt.plot(time_axis, grand_average, linewidth=2, color=colors(j), label=f'{int(currentStimDuration)} ms')

    plt.ylabel('WBF (Hz)', color='white')
    plt.xlabel('Time (s)', color='white')
    plt.title('Summary of All Stim Durations', color='white')
    plt.legend(loc='northeast', frameon=False, labelcolor='white')

    # Save the final summary figure
    save_dir = 'D:\\Yichen\\Plots'
    final_summary_file_name = 'Final_Summary_WBF_plot_All_Durations.png'
    plt.savefig(os.path.join(save_dir, final_summary_file_name), dpi=300, bbox_inches='tight', facecolor='black')
    plt.close()

if __name__ == '__main__':
    # Prompt the user to select the .mat files to process
    root = Tk()
    root.withdraw()
    files = filedialog.askopenfilenames(title="Select the .mat files to include", filetypes=[("MAT files", "*.mat")])
    dataFolder = os.path.dirname(files[0])

    summary_data = process_files(dataFolder, files)
    uniqueStimDurations = sorted(set([float(dur) for file_data in summary_data.values() for dur in file_data.keys()]))

    create_summary_plots(summary_data, uniqueStimDurations)
    create_final_summary_plot(summary_data, uniqueStimDurations)
